In [1]:
import pandas as pd
import numpy as np

In [5]:
# Load data
df = pd.read_excel('perfect_final_data_for_multilevel_analysis.xlsx')

# Display basic info
print(df.shape)
print(df.columns.tolist())
print(df.info())
print(df.head())

(8502, 41)
['V001', 'V002', 'V012', 'V013', 'V024', 'V025', 'V106', 'V113', 'V130', 'V157', 'V158', 'V159', 'V190', 'V201', 'V212', 'V218', 'V221', 'V364', 'V404', 'V445', 'V447', 'V501', 'V502', 'V701', 'V705', 'V714', 'V730', 'BMI_fixed', 'Media_acccess', 'husband_education', 'contraceptive_use', 'Number_of_children', 'Living_children', 'Age_of_husband', 'profession_of_husband', 'Birth_Interval', 'Drinking_water', 'Wealth_Status', 'Religion_status', 'BMI', 'Respondent_Age_first_birth']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8502 entries, 0 to 8501
Data columns (total 41 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   V001                        8502 non-null   int64  
 1   V002                        8502 non-null   int64  
 2   V012                        8502 non-null   int64  
 3   V013                        8502 non-null   int64  
 4   V024                        8502 non-null   int64  
 

In [7]:
df2 = df[df['V447'] == 0].copy()
df2.shape

(8502, 41)

In [8]:
df = df.drop(columns=['V447'])

In [10]:
feature_cols = [
    'V012', 'V024', 'V025', 'V106','V404','V502','V501','V714',
    'Drinking_water', 'Media_acccess', 'Wealth_Status',
    'Number_of_children', 'Living_children', 'Birth_Interval',
    'contraceptive_use', 'husband_education', 'profession_of_husband',
    'Age_of_husband', 'Respondent_Age_first_birth'
]

# Make sure all columns exist
existing_cols = [col for col in feature_cols if col in df.columns]
missing = set(feature_cols) - set(existing_cols)
if missing:
    print("Missing columns:", missing)
    # You'll need to create them or drop them
    # For now, we'll use only existing ones
    feature_cols = existing_cols

X = df[feature_cols].copy()
y = df['BMI'].copy()   # categorical target

# Verify target values
print(y.value_counts().sort_index())

BMI
1     806
2    4469
3    2533
4     694
Name: count, dtype: int64


In [11]:
print(X.isnull().sum())

V012                          0
V024                          0
V025                          0
V106                          0
Drinking_water                0
Religion_status               0
Media_acccess                 0
Wealth_Status                 0
Number_of_children            0
Living_children               0
Birth_Interval                0
contraceptive_use             0
husband_education             0
profession_of_husband         0
Age_of_husband                0
Respondent_Age_first_birth    0
dtype: int64


# encoding for nominal categories 

# Train Baseline Models

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

nominal_cols = ['V024', 'Drinking_water', 'profession_of_husband','Religion_status']   # adjust if needed

preprocessor = ColumnTransformer(
    transformers=[
        ('nom', OneHotEncoder(drop='first', handle_unknown='ignore'), nominal_cols)
    ],
    remainder='passthrough'
)

In [25]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', None)   # placeholder, we'll set it later
])

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Set classifier
pipeline.set_params(classifier=LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))

# Train
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

C:\Users\ASUS\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\ASUS\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^

Logistic Regression Accuracy: 0.25631981187536745
              precision    recall  f1-score   support

           1       0.16      0.54      0.25       161
           2       0.57      0.20      0.30       894
           3       0.34      0.21      0.26       507
           4       0.12      0.44      0.19       139

    accuracy                           0.26      1701
   macro avg       0.30      0.35      0.25      1701
weighted avg       0.43      0.26      0.27      1701



C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [27]:
from sklearn.ensemble import RandomForestClassifier

pipeline.set_params(classifier=RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
pipeline.fit(X_train, y_train)
y_pred_rf = pipeline.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           1       0.17      0.04      0.07       161
           2       0.55      0.77      0.64       894
           3       0.37      0.26      0.31       507
           4       0.08      0.03      0.04       139

    accuracy                           0.49      1701
   macro avg       0.29      0.28      0.27      1701
weighted avg       0.42      0.49      0.44      1701



In [31]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import pandas as pd

# Remap target
y_train_mapped = y_train.map({1:0, 2:1, 3:2, 4:3})
y_test_mapped = y_test.map({1:0, 2:1, 3:2, 4:3})

# Set classifier in pipeline
pipeline.set_params(classifier=XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
))

# Fit on mapped targets
pipeline.fit(X_train, y_train_mapped)

# Predict
y_pred_mapped = pipeline.predict(X_test)

# Map back to original labels
inverse_map = {0:1, 1:2, 2:3, 3:4}
y_pred_original = pd.Series(y_pred_mapped).map(inverse_map)

# Evaluate
print(classification_report(y_test, y_pred_original))

              precision    recall  f1-score   support

           1       0.21      0.05      0.08       161
           2       0.56      0.78      0.65       894
           3       0.41      0.29      0.34       507
           4       0.27      0.09      0.14       139

    accuracy                           0.51      1701
   macro avg       0.36      0.30      0.30      1701
weighted avg       0.46      0.51      0.46      1701



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train_enc, y_train)   # you need X_train_enc from preprocessor
# Then plot